# DE séquentiel (CPU) + DE CUDA basique (GPU) — Kaggle

Version **Kaggle Notebooks** (GPU gratuit, ~30h/semaine, pas de carte bancaire requise) du notebook DE — même code C++/CUDA que la version Colab, seule la partie upload/téléchargement de fichiers change (Kaggle n'a pas `google.colab.files`).

## Avant de lancer quoi que ce soit

1. Dans le panneau de droite (⚙️ **Settings**) : **Accelerator → GPU T4 x2** (ou P100).
2. Toujours dans **Settings** : **Internet → On** (pas strictement nécessaire ici, mais évite des surprises).
3. Ajoute les 4 fichiers du projet via le panneau de droite : **Add Data → Upload → New Dataset**, sélectionne `main.cpp`, `kernel.h`, `kernel.cpp`, `kernel.cu`, donne un nom au dataset (ex. `de-cuda-sources`), puis clique **Create**. Kaggle les monte en lecture seule sous `/kaggle/input/<nom-du-dataset>/`.

Le reste (compilation, exécution, résultats) fonctionne à l'identique de la version Colab.

In [ ]:
import os
import shutil
import subprocess
import re
from pathlib import Path

print('=== Vérification du GPU ===')
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        print('⚠️  nvidia-smi a échoué — vérifie Settings → Accelerator → GPU T4 x2.')
    else:
        print(result.stdout)
        print('✅ GPU détecté.')
except FileNotFoundError:
    print("⚠️  nvidia-smi introuvable : l'accélérateur GPU n'est probablement pas activé.")
    print('    Panneau de droite → Settings → Accelerator → GPU T4 x2, puis relance cette cellule.')

## 1. Vérifier `nvcc`

Les images GPU Kaggle embarquent normalement un toolkit CUDA préinstallé — même logique que sur Colab, pas de réinstallation manuelle.

In [ ]:
print('=== Recherche de nvcc ===')
nvcc = shutil.which('nvcc')

if not nvcc:
    candidates = sorted(Path('/usr/local').glob('cuda*/bin/nvcc'), reverse=True)
    for candidate in candidates:
        if candidate.exists():
            cuda_bin = str(candidate.parent)
            os.environ['PATH'] = cuda_bin + ':' + os.environ['PATH']
            nvcc = shutil.which('nvcc')
            if nvcc:
                print('✅ nvcc trouvé après ajout au PATH :', nvcc)
                break

if nvcc:
    print('✅ nvcc :', nvcc)
    subprocess.run([nvcc, '--version'], check=False)
else:
    raise RuntimeError(
        "❌ nvcc introuvable. Vérifie Settings → Accelerator → GPU T4 x2, puis relance cette cellule."
    )

## 2. Localiser les 4 fichiers uploadés

Kaggle monte les datasets en lecture seule sous `/kaggle/input/<nom>/` — on les cherche là (peu importe le nom exact du dataset), puis on les copie dans `/kaggle/working/DE_CUDA/` (seul dossier réellement accessible en écriture).

In [ ]:
PROJECT_DIR = Path('/kaggle/working/DE_CUDA')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

required = ['main.cpp', 'kernel.h', 'kernel.cpp', 'kernel.cu']
search_root = Path('/kaggle/input')

print('=== Recherche sous /kaggle/input ===')
missing = []
for filename in required:
    matches = list(search_root.rglob(filename))
    if matches:
        shutil.copy2(matches[0], PROJECT_DIR / filename)
        print(f'✅ {filename} trouvé dans {matches[0]}')
    else:
        print(f'❌ {filename} introuvable sous /kaggle/input')
        missing.append(filename)

if missing:
    raise FileNotFoundError(
        'Fichiers manquants : ' + ', '.join(missing) +
        " — vérifie que le dataset est bien attaché (panneau de droite → Add Data) et que les noms de fichiers sont exacts."
    )

print('\nContenu de', PROJECT_DIR, ':')
for p in sorted(PROJECT_DIR.iterdir()):
    print(' -', p.name, f'({p.stat().st_size} octets)')

## 3. Compilation

Identique à la version Colab : les 4 fichiers ensemble avec `nvcc`, `-lcurand` pour les kernels DE GPU.

In [ ]:
os.chdir(PROJECT_DIR)

command = [
    nvcc,
    '-O2',
    '-std=c++17',
    '-diag-suppress=546',   # transfer of control bypasses initialization (switch objectif, kernel.cu)
    '-o', 'programde',
    'main.cpp',
    'kernel.cpp',
    'kernel.cu',
    '-lcurand',
]

print('Commande :')
print(' '.join(command))

compile_result = subprocess.run(command, capture_output=True, text=True)
print('\n--- stdout ---')
print(compile_result.stdout)
print('\n--- stderr ---')
print(compile_result.stderr)

if compile_result.returncode != 0:
    raise RuntimeError('❌ Compilation échouée (voir stderr ci-dessus).')

print('✅ Compilation réussie :', PROJECT_DIR / 'programde')

## 4. Vérifier l'exécutable et son usage

In [ ]:
exe = PROJECT_DIR / 'programde'
print('Executable :', exe, '—', exe.stat().st_size, 'octets')
!/kaggle/working/DE_CUDA/programde   # sans arguments : affiche l'usage attendu (mode cpu|gpu)

## 5. Fonction `run_de` (avec mode) et parsing du résultat

In [ ]:
OUTPUT_RE = re.compile(r'DE-(GPU-OPT|GPU-FUSED|GPU|CPU)\s+([\d.]+)\s+(-?[\d.eE+-]+)\s+(\d+)')

def run_de(dim, population, objective=1, seed=42, mode='cpu', timeout=300):
    if population < 4:
        raise ValueError('population doit être >= 4 (contrainte DE/rand/1).')
    if mode not in ('cpu', 'gpu', 'gpu-fused', 'gpu-opt'):
        raise ValueError("mode doit être 'cpu', 'gpu', 'gpu-fused' ou 'gpu-opt'")
    if mode == 'gpu-opt' and objective == 0:
        raise ValueError("mode gpu-opt ne supporte pas objectif=0 (Levy)")

    cmd = [str(exe), str(dim), str(population), str(objective), str(seed), mode]
    print('Commande :', ' '.join(cmd))
    try:
        result = subprocess.run(
            cmd, cwd=PROJECT_DIR, capture_output=True, text=True, timeout=timeout
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError(f'⏱️ Timeout ({timeout}s) dépassé pour dim={dim}, pop={population}, mode={mode}')

    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print('--- stderr ---')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f'Programme terminé avec le code {result.returncode} '
            f'(dim={dim}, pop={population}, objectif={objective}, seed={seed}, mode={mode})'
        )

    match = OUTPUT_RE.search(result.stdout)
    if not match:
        raise RuntimeError(f'Sortie inattendue, impossible de parser :\n{result.stdout}')

    return {
        'dimension': dim, 'population': population, 'objective': objective,
        'seed': seed, 'mode': match.group(1).lower(),
        'time_sec': float(match.group(2)), 'minimum': float(match.group(3)), 'fes': int(match.group(4)),
    }

run_de(10, 50, 1, 42, mode='cpu')


## 6. Premier test GPU

Petite config d'abord (D=10, NP=50) — si ça plante, la sortie `stderr` du binaire (macro `CUDA_CHECK`) indique directement le fichier/ligne et l'erreur CUDA.

In [ ]:
run_de(10, 50, 1, 42, mode='gpu')

## 7. Comparer CPU vs GPU sur les mêmes configurations

Générateurs aléatoires différents (host `rand()` vs device `curand`) → résultats non identiques au bit près même à seed égale, c'est attendu. On vérifie que le GPU converge vers les mêmes ordres de grandeur que le CPU.

In [ ]:
OBJ_NAMES = {0: 'Levy', 1: 'Rastrigin', 2: 'Rosenbrock', 3: 'Griewank', 4: 'Sphere'}

comparison = []
for objective in [1, 2, 3, 4]:
    print(f'\n### {OBJ_NAMES[objective]} (objectif={objective}) ###')
    comparison.append(run_de(10, 50, objective, 42, mode='cpu'))
    comparison.append(run_de(10, 50, objective, 42, mode='gpu'))

import pandas as pd
df_compare = pd.DataFrame(comparison)
df_compare['objective_name'] = df_compare['objective'].map(OBJ_NAMES)
df_compare

## 8. Montée en charge GPU (D=50/NP=100, D=100/NP=500)

In [ ]:
run_de(50, 100, 1, 42, mode='gpu')
run_de(100, 500, 1, 42, mode='gpu')

## 9. Sauvegarder les résultats

Kaggle n'a pas de `files.download()` : tout ce qui est écrit dans `/kaggle/working/` apparaît automatiquement dans le panneau **Output** à droite (bouton de téléchargement), et reste disponible même après la fin de la session. `FileLink` ci-dessous donne aussi un lien cliquable direct dans le notebook.

In [ ]:
from IPython.display import FileLink

csv_path = PROJECT_DIR / 'comparaison_cpu_gpu.csv'
df_compare.to_csv(csv_path, index=False)
print('✅ Résultats sauvegardés :', csv_path)
print('   (visible aussi dans le panneau Output à droite, ou via le lien ci-dessous)')

FileLink(str(csv_path.relative_to('/kaggle/working')), result_html_prefix='Télécharger : ')

## 10. Version fusionnée (architecture inspirée de cudaDEᵢ, Figure 2)

`mode='gpu-fused'` utilise `cuda_de_fused` : kernel(I)+kernel(E) → kernel(IE), kernel(M)+kernel(C)+kernel(E)+
kernel(R) → kernel(MCER). 2 lancements de kernels par génération au lieu de 5, et le trial vector ne transite
plus par la mémoire globale (buffers `d_V`/`d_U` supprimés). Attendu : gain net sur les configurations où le
coût de lancement des kernels dominait (D élevé, NP faible), là où la version de base perdait contre le CPU.


In [ ]:
run_de(10, 50, 1, 42, mode='gpu-fused')


## 11. Comparer les 3 modes sur les configurations où la version de base perdait

Rappel des résultats de la version de base (`gpu`) : à D=100, NP=50, le GPU perdait nettement contre le CPU
(overhead de lancement des kernels dominant). On reteste ces mêmes configurations avec `gpu-fused` pour voir
si la fusion comble l'écart.


In [ ]:
fused_comparison = []
configs = [(100, 50), (100, 100), (100, 500)]
for dim, pop in configs:
    print(f'\n=== D={dim}, NP={pop} (Rastrigin) ===')
    fused_comparison.append(run_de(dim, pop, 1, 42, mode='cpu'))
    fused_comparison.append(run_de(dim, pop, 1, 42, mode='gpu'))
    fused_comparison.append(run_de(dim, pop, 1, 42, mode='gpu-fused'))

df_fused = pd.DataFrame(fused_comparison)
df_fused


## 12. Version optimisée (mémoire partagée + streams + occupancy)

`mode='gpu-opt'` utilise `cuda_de_optimized` : mapping 1 bloc = 1 individu / 1 thread = 1 gène pour
kernel(IE) et kernel(MCER) (mémoire partagée pour r1/r2/r3/j_rand partagés + réduction parallèle de la
fitness), kernel(P) et kernel(MCER) recouverts via 2 streams CUDA (kernel(P) de la génération suivante
préparé pendant que kernel(MCER) de la génération courante tourne), et `threadsPerBlock` de kernel(P)
déterminé par `cudaOccupancyMaxPotentialBlockSize` au lieu d'une valeur fixe.

⚠️ Ne supporte pas objectif=0 (Levy — hors protocole officiel). Le programme refuse proprement avec un
message clair si on essaie.

**Point de vigilance** : cette version n'a jamais été compilée ni exécutée avant ton premier test — relis
bien la sortie `stderr` (macro `CUDA_CHECK`) en cas d'échec, et compare le `minimum` obtenu à celui de
`gpu-fused` sur la même config : les valeurs peuvent différer légèrement (l'ordre de consommation des
tirages aléatoires curand n'est pas strictement identique entre les deux architectures), mais doivent rester
du même ordre de grandeur.


In [ ]:
run_de(10, 50, 1, 42, mode='gpu-opt')


## 13. Comparer les 4 modes sur les configurations les plus dures pour le GPU

Rappel : `gpu` et `gpu-fused` perdaient tous les deux contre le CPU à D=100/NP=50. On reteste avec `gpu-opt`
pour voir si la mémoire partagée + les streams + l'occupancy comblent (ou inversent) cet écart.


In [ ]:
opt_comparison = []
configs = [(100, 50), (100, 100), (100, 500)]
for dim, pop in configs:
    print(f'\n=== D={dim}, NP={pop} (Rastrigin) ===')
    opt_comparison.append(run_de(dim, pop, 1, 42, mode='cpu'))
    opt_comparison.append(run_de(dim, pop, 1, 42, mode='gpu'))
    opt_comparison.append(run_de(dim, pop, 1, 42, mode='gpu-fused'))
    opt_comparison.append(run_de(dim, pop, 1, 42, mode='gpu-opt'))

df_opt = pd.DataFrame(opt_comparison)
df_opt
